### Pre-Processing SQUID data

How to use: 

1. run in osld enviroment - will need to 'pip install PyQt6' 'pip install pyqt6 pyqt6-tools' 
2. put in your data 
3. look at the psds to determine bad channels
4. unmark bad channels 
5. go through data and mark/unmark bad segments 
6. close mne interactive figure 

In [1]:
#Importing necessary libraries
import os
os.chdir("/home/anna-beer/Documents/anna_phd/Canonical-HMM-Networks") #sets working directory to the repo, so that all imports work correctly
print(os.getcwd())
import numpy as np
from pathlib import Path
import matplotlib
import matplotlib.pyplot as plt
import mne
# matplotlib.use('Qt5Agg')
#matplotlib.use('qtagg')
# print(matplotlib.get_backend())
from osl_dynamics.meeg import amm, preproc

import pyqtgraph as pg
# Disable OpenGL in PyQtGraph to prevent Linux GPU crashes
pg.setConfigOption('useOpenGL', False)
mne.viz.set_browser_backend('qt')
%matplotlib qt
import gc


/home/anna-beer/Documents/anna_phd/Canonical-HMM-Networks
Using qt as 2D backend.


In [2]:
def preprocess_ctf(rawfile, preprocfile, figuresdir):

    ''' 
    1.Detects bad channels automatically and also allows for manual editing of selected bad channels. 
    2.detects bad segments and allows for review and modification (adding/removing) of the bad segments detected.
    3.Then saves out a fif file with the bad channels and segments annotated.
    '''

    #1. LOAD IN THE RAW DATA 
    print('opening raw file')
    raw = mne.io.read_raw_ctf(rawfile, preload=True) #loading in the raw .fif data
    print('done reading in raw data')

    # --------------------------------------------------------------------------------------------------------------

    #2. PRE-PROCESSING
    #raw = raw.set_channel_types({'Rig': 'stim', 'Lef': 'stim'})
    #raw, amm_info = amm.apply_amm(raw)
    raw.apply_gradient_compensation(3)
    raw = raw.resample(sfreq=250)
    raw = raw.filter(l_freq=1, h_freq=120, method="iir", iir_params={"order": 5, "ftype": "butter"})
    raw = raw.notch_filter([50, 100, 57, 109])

    # --------------------------------------------------------------------------------------------------------------

    #3. CROPPING THE DATASET TO -1s BEFORE THE FIRST EVENT AND +20s AFTER THE LAST EVENT
    channels = ['Right_trial', 'Left_trial']
    all_events = []
    for ch in channels:
        events = mne.find_events(raw, stim_channel=ch, shortest_event=1)
        if events.size > 0:
            all_events.append(events)
            
    all_events = np.vstack(all_events)
    # Get the first and last sample across both channels
    first_sample = all_events[:, 0].min()
    last_sample = all_events[:, 0].max()
    # Add 20 seconds after last event
    sfreq = raw.info['sfreq']
    extra_samples_1s = int(1 * sfreq)
    extra_samples_20s = int(20 * sfreq)

    max_time = raw.times[-1]
    tmin = (first_sample - extra_samples_1s) / sfreq
    tmax = (last_sample + extra_samples_20s) / sfreq

    tmax = min(tmax, max_time)

    raw = raw.crop(tmin=tmin, tmax=tmax)
    # --------------------------------------------------------------------------------------------------------------

    #4. DETECT BAD CHANNELS 
    raw = preproc.detect_bad_channels(raw, picks="mag")
    #fig1 = preproc.plot_channel_stds(raw)
    #fig2 = raw.plot_psd()
    fig3 = raw.compute_psd(picks='mag').plot()
    plt.show(block=True)
    
    # --------------------------------------------------------------------------------------------------------------

    #5. DETECT BAD SEGMENTS
    raw = preproc.detect_bad_segments(raw, picks="mag")
    raw = preproc.detect_bad_segments(raw, picks="mag", mode="diff")
    raw.plot(block=True, picks='mag', n_channels=275, duration=15) #275 total meg channels but 29 ref channels 

    # --------------------------------------------------------------------------------------------------------------

    #6. SAVE PREPROCESSED DATA 
    raw.save(preprocfile, overwrite=True)

    #Sav preproc info 
    fig4 = raw.compute_psd(picks='mag').plot(show=False)  # do not show yet
    fig4.savefig(f"{figuresdir}/psd_markedbadchannels.png", dpi=600) 
    plt.close(fig4)  # optional, closes figure to free memory
    fig5 = raw.plot_psd(picks='mag',show=False)  # do not show yet
    fig5.savefig(f"{figuresdir}/psd_nobadchannels.png", dpi=600) 
    plt.close(fig5)  # optional, closes figure to free memory

    return None

In [3]:
subjects = ['016']#, '003', '004', '005', '006', '007', '009', '010', '011', '012', '013', '014', '015', '016']
sessions = ["003", "004"]
tasks = ["braille"]
runs = ["001"]


base = "/rdrives/DRS-foundation-brain/zoe_data/BIDS"
deriv = f"{base}/derivatives_anna_01042026"
preprocessed_dir = f"{deriv}/preprocessed"
os.makedirs(deriv, exist_ok=True)


for subject in subjects:
    for session in sessions:
        for task in tasks:
            for run in runs:

                id = f"sub-{subject}_ses-{session}_task-{task}_run-{run}"
                raw_file = f"{base}/sub-{subject}/ses-{session}/meg/{id}_meg.ds" 
                preproc_dir = f"{preprocessed_dir}/{id}"
                os.makedirs(preproc_dir, exist_ok=True)
                preproc_file = f"{preproc_dir}/{id}_preproc-raw.fif" 
                figures_dir = f"{preproc_dir}/figures"
                os.makedirs(figures_dir, exist_ok=True)

                

                preprocess_ctf(raw_file, preproc_file, figures_dir)


opening raw file
ds directory : /rdrives/DRS-foundation-brain/zoe_data/BIDS/sub-016/ses-003/meg/sub-016_ses-003_task-braille_run-001_meg.ds
    res4 data read.
    hc data read.
    Separate EEG position data file not present.
    Quaternion matching (desired vs. transformed):
       2.16   70.04    0.00 mm <->    2.16   70.04    0.00 mm (orig :  -52.33   46.96 -259.87 mm) diff =    0.000 mm
      -2.16  -70.04    0.00 mm <->   -2.16  -70.04   -0.00 mm (orig :   40.73  -57.33 -249.68 mm) diff =    0.000 mm
     103.54    0.00    0.00 mm <->  103.54   -0.00    0.00 mm (orig :   67.42   66.09 -238.09 mm) diff =    0.000 mm
    Coordinate transformations established.
    Reading digitizer points from ['/rdrives/DRS-foundation-brain/zoe_data/BIDS/sub-016/ses-003/meg/sub-016_ses-003_task-braille_run-001_meg.ds/17611_20240415_01.pos']...
    Polhemus data for 3 HPI coils added
    Device coordinate locations for 3 HPI coils added
Picked positions of 64 EEG channels from channel info
    64 E

/tmp/ipykernel_474698/405070315.py:56: RuntimeWarning: (X, Y) fit (13.2, 15.4) more than 20 mm from head frame origin
  fig3 = raw.compute_psd(picks='mag').plot()



Bad segment detection
---------------------
Modality: mag
Mode: None
Metric: std
Significance level: 0.05
Maximum fraction: 0.1
Found 2 bad segments: 4.0/1310.9 seconds rejected (0.3%)

Bad segment detection
---------------------
Omitting 1000 of 327720 (0.31%) samples, retaining 326720 (99.69%) samples.
Modality: mag
Mode: diff
Metric: std
Significance level: 0.05
Maximum fraction: 0.1
Found 4 bad segments: 14.0/1310.9 seconds rejected (1.1%)
Channels marked as bad:
[np.str_('BG1-4123'), np.str_('BG2-4123'), np.str_('BG3-4123'), np.str_('BP1-4123'), np.str_('BP2-4123'), np.str_('BP3-4123'), np.str_('BR1-4123'), np.str_('BR2-4123'), np.str_('BR3-4123'), np.str_('G11-4123'), np.str_('G12-4123'), np.str_('G13-4123'), np.str_('G22-4123'), np.str_('G23-4123'), np.str_('P11-4123'), np.str_('P12-4123'), np.str_('P13-4123'), np.str_('P23-4123'), np.str_('Q11-4123'), np.str_('Q22-4123'), np.str_('Q23-4123'), np.str_('R11-4123'), np.str_('R12-4123'), np.str_('R13-4123'), np.str_('R22-4123'), n

/tmp/ipykernel_474698/405070315.py:72: RuntimeWarning: (X, Y) fit (13.2, 15.4) more than 20 mm from head frame origin
  fig4 = raw.compute_psd(picks='mag').plot(show=False)  # do not show yet


NOTE: plot_psd() is a legacy function. New code should use .compute_psd().plot().
Removing 5 compensators from info because not all compensation channels were picked.
Effective window size : 8.192 (s)
Plotting power spectral density (dB=True).


/tmp/ipykernel_474698/405070315.py:75: RuntimeWarning: (X, Y) fit (13.2, 15.4) more than 20 mm from head frame origin
  fig5 = raw.plot_psd(picks='mag',show=False)  # do not show yet


qt.qpa.wayland: There are no outputs - creating placeholder screen
